# Benchmark 1 — Influence conjointe de $\alpha$ et $\beta$ du critère d'Armijo sur la descente de gradient

In [ ]:
import sys

sys.path.insert(0, "..")

import os
import warnings
import numpy as np
import matplotlib.pyplot as plt
from matplotlib.colors import LogNorm
from joblib import Parallel, delayed
from tqdm.auto import tqdm

from src.test_functions import create_system, Rosenbrock

In [ ]:
kappa_values = [5, 50, 500]
gamma_values = [1, 10, 100]

seeds = list(range(15))
x0_domain = (-2, 2)

alpha_values = np.linspace(0.01, 0.49, 15)
beta_values = np.linspace(0.01, 0.99, 15)

eps = 1e-6
max_iter = 10000
dim = 10

In [ ]:
class FuncCounter:
    def __init__(self, f):
        self.f = f
        self.count = 0

    def __call__(self, x):
        self.count += 1
        return self.f(x)

In [ ]:
def grad_quadraticn(x, A, b):
    return A @ x - b


def grad_rosenbrock(x, gamma=100):
    return np.array(
        [
            2 * (x[0] - 1) + 4 * gamma * x[0] * (x[0] ** 2 - x[1]),
            -2 * gamma * (x[0] ** 2 - x[1]),
        ]
    )

In [ ]:
def armijo_fast(xk, fk, f, dir_x, gk_dot_d, alpha, beta):
    eta = 1.0
    xt = xk + eta * dir_x
    ft = f(xt)
    while ft > fk + alpha * eta * gk_dot_d:
        eta *= beta
        xt = xk + eta * dir_x
        ft = f(xt)
    return eta, ft, xt

In [ ]:
def desc_grad_opti_ab(f, grad_f, x0, alpha, beta, eps=1e-6, max_iter=10000):
    xk = x0.copy()
    positions = [xk]
    fk = f(xk)
    converged = False
    for _ in range(max_iter):
        gk = grad_f(xk)
        if np.linalg.norm(gk) < eps:
            converged = True
            break
        dir_x = -gk
        eta, fk1, xk1 = armijo_fast(xk, fk, f, dir_x, np.dot(gk, dir_x), alpha, beta)
        positions.append(xk1)
        xk = xk1
        fk = fk1
    return np.array(positions), converged

In [ ]:
def _run_quad(ki, ai, bi, si, alpha, beta, kappa, seed, dim, eps, max_iter):
    A, b = create_system(dim, cond=kappa, seed=seed)
    x_exact = np.linalg.solve(A, b)
    f = lambda x, A=A, b=b: (x.T @ A @ x) / 2 - b.T @ x
    gf = lambda x, A=A, b=b: grad_quadraticn(x, A, b)
    fc = FuncCounter(f)
    with warnings.catch_warnings():
        warnings.simplefilter("ignore")
        res, converged = desc_grad_opti_ab(
            fc, gf, np.zeros(dim), alpha, beta, eps, max_iter
        )
    n_iter = len(res) - 1
    return (
        ki,
        ai,
        bi,
        si,
        [n_iter, fc.count, np.linalg.norm(res[-1] - x_exact), int(converged)],
    )


def _run_rosen(gi, ai, bi, si, alpha, beta, gamma, x0, eps, max_iter):
    f = lambda x, g=gamma: Rosenbrock(x, gamma=g)
    gf = lambda x, g=gamma: grad_rosenbrock(x, gamma=g)
    fc = FuncCounter(f)
    with warnings.catch_warnings():
        warnings.simplefilter("ignore")
        res, converged = desc_grad_opti_ab(fc, gf, x0, alpha, beta, eps, max_iter)
    n_iter = len(res) - 1
    return (gi, ai, bi, si, [n_iter, fc.count, int(converged)])

In [ ]:
def run_benchmark(
    alpha_values,
    beta_values,
    seeds,
    kappa_values,
    gamma_values,
    dim,
    eps,
    max_iter,
    x0_domain,
    n_jobs=-1,
):
    n_alpha, n_beta, n_seeds = len(alpha_values), len(beta_values), len(seeds)

    tasks_quad = [
        (
            ki,
            ai,
            bi,
            si,
            alpha_values[ai],
            beta_values[bi],
            kappa,
            seeds[si],
            dim,
            eps,
            max_iter,
        )
        for ki, kappa in enumerate(kappa_values)
        for ai in range(n_alpha)
        for bi in range(n_beta)
        for si in range(n_seeds)
    ]
    results_quad = np.full((len(kappa_values), n_alpha, n_beta, n_seeds, 4), np.nan)
    for ki, ai, bi, si, val in tqdm(
        Parallel(n_jobs=n_jobs, return_as="generator")(
            delayed(_run_quad)(*t) for t in tasks_quad
        ),
        total=len(tasks_quad),
        desc="quadraticn_",
    ):
        results_quad[ki, ai, bi, si] = val

    rng = np.random.default_rng(42)
    x0_list = rng.uniform(x0_domain[0], x0_domain[1], size=(n_seeds, 2))

    tasks_rosen = [
        (
            gi,
            ai,
            bi,
            si,
            alpha_values[ai],
            beta_values[bi],
            gamma,
            x0_list[si],
            eps,
            max_iter,
        )
        for gi, gamma in enumerate(gamma_values)
        for ai in range(n_alpha)
        for bi in range(n_beta)
        for si in range(n_seeds)
    ]
    results_rosen = np.full((len(gamma_values), n_alpha, n_beta, n_seeds, 3), np.nan)
    for gi, ai, bi, si, val in tqdm(
        Parallel(n_jobs=n_jobs, return_as="generator")(
            delayed(_run_rosen)(*t) for t in tasks_rosen
        ),
        total=len(tasks_rosen),
        desc="Rosenbrock",
    ):
        results_rosen[gi, ai, bi, si] = val

    return results_quad, results_rosen

In [ ]:
RESULTS_QUAD_PATH = "../results_quad_main.npy"
RESULTS_ROSEN_PATH = "../results_rosen_main.npy"

if os.path.exists(RESULTS_QUAD_PATH) and os.path.exists(RESULTS_ROSEN_PATH):
    results_quad_main = np.load(RESULTS_QUAD_PATH)
    results_rosen_main = np.load(RESULTS_ROSEN_PATH)
    print(
        f"Chargé depuis le cache : quad {results_quad_main.shape}, rosen {results_rosen_main.shape}"
    )
else:
    results_quad_main, results_rosen_main = run_benchmark(
        alpha_values,
        beta_values,
        seeds,
        kappa_values,
        gamma_values,
        dim=dim,
        eps=eps,
        max_iter=max_iter,
        x0_domain=x0_domain,
    )
    np.save(RESULTS_QUAD_PATH, results_quad_main)
    np.save(RESULTS_ROSEN_PATH, results_rosen_main)
    print("Phase principale terminée, résultats sauvegardés.")

In [ ]:
ITER, FEVAL, ERR_Q, CONV_Q = 0, 1, 2, 3
ITER_R, FEVAL_R, CONV_R = 0, 1, 2

rqm = results_quad_main
rrm = results_rosen_main


def conv_rate(res, ci):
    return res[..., ci].mean(axis=-1)


def med_over_conv(res, mi, ci, conv_threshold=0.5):
    """Médiane de la métrique mi sur les seeds convergés, NaN si taux trop faible."""
    cr = conv_rate(res, ci)
    mask = res[..., ci].astype(bool)
    vals = np.where(mask, res[..., mi].astype(float), np.nan)
    with warnings.catch_warnings():
        warnings.simplefilter("ignore", RuntimeWarning)
        meds = np.nanmedian(vals, axis=-1)
    return np.where(cr >= conv_threshold, meds, np.nan)

In [ ]:
cr_q = np.stack([conv_rate(rqm, CONV_Q)[ki] for ki in range(len(kappa_values))])
it_q = np.stack(
    [med_over_conv(rqm, ITER, CONV_Q)[ki] for ki in range(len(kappa_values))]
)
fv_q = np.stack(
    [med_over_conv(rqm, FEVAL, CONV_Q)[ki] for ki in range(len(kappa_values))]
)

cr_r = np.stack([conv_rate(rrm, CONV_R)[gi] for gi in range(len(gamma_values))])
it_r = np.stack(
    [med_over_conv(rrm, ITER_R, CONV_R)[gi] for gi in range(len(gamma_values))]
)
fv_r = np.stack(
    [med_over_conv(rrm, FEVAL_R, CONV_R)[gi] for gi in range(len(gamma_values))]
)

A_grid, B_grid = np.meshgrid(alpha_values, beta_values, indexing="ij")

print("Taux de convergence min (quad) :", cr_q.min())
print("Taux de convergence min (rosen):", cr_r.min())

In [ ]:
OUT = "../figures/benchmark1"
os.makedirs(OUT, exist_ok=True)

plt.rcParams["mathtext.fontset"] = "cm"
plt.rcParams["font.size"] = 11
plt.rcParams["axes.titlesize"] = 12
plt.rcParams["axes.labelsize"] = 11

In [ ]:
fig, axes = plt.subplots(2, 3, figsize=(14, 8), constrained_layout=True)
cmap = plt.get_cmap("viridis_r").copy()
cmap.set_bad("#222222")

vmin_q = max(np.nanmin(it_q), 1)
vmax_q = np.nanmax(it_q)
vmin_r = max(np.nanmin(it_r), 1) if np.any(np.isfinite(it_r)) else 1
vmax_r = np.nanmax(it_r) if np.any(np.isfinite(it_r)) else 10

ims_q, ims_r = [], []
for ki, k in enumerate(kappa_values):
    ax = axes[0, ki]
    im = ax.pcolormesh(
        A_grid,
        B_grid,
        np.ma.masked_invalid(it_q[ki]),
        cmap=cmap,
        norm=LogNorm(vmin=vmin_q, vmax=vmax_q),
        shading="nearest",
    )
    ims_q.append(im)
    ax.set_title(rf"Quadratique, $\kappa={k}$")
    ax.set_xlabel(r"$\alpha$")
    ax.set_ylabel(r"$\beta$")

for gi, g in enumerate(gamma_values):
    ax = axes[1, gi]
    if np.all(np.isnan(it_r[gi])):
        ax.set_facecolor("#222222")
        ax.set_title(rf"Rosenbrock, $\gamma={g}$ (échec)")
    else:
        im = ax.pcolormesh(
            A_grid,
            B_grid,
            np.ma.masked_invalid(it_r[gi]),
            cmap=cmap,
            norm=LogNorm(vmin=vmin_r, vmax=vmax_r),
            shading="nearest",
        )
        ims_r.append(im)
        ax.set_title(rf"Rosenbrock, $\gamma={g}$")
    ax.set_xlabel(r"$\alpha$")
    ax.set_ylabel(r"$\beta$")
    ax.set_xlim(alpha_values.min(), alpha_values.max())
    ax.set_ylim(beta_values.min(), beta_values.max())

cb_q = fig.colorbar(ims_q[-1], ax=axes[0, :], shrink=0.85, pad=0.02)
cb_q.set_label("Médiane des itérations (log)")
if ims_r:
    cb_r = fig.colorbar(ims_r[-1], ax=axes[1, :], shrink=0.85, pad=0.02)
    cb_r.set_label("Médiane des itérations (log)")

fig.suptitle("Nombre d'itérations externes", fontsize=14, weight="bold")
fig.legend(
    handles=[
        plt.Line2D([0], [0], color="#222222", marker="s", linestyle="", markersize=8)
    ],
    labels=["Zones noires = non-convergence"],
    loc="upper right",
)
fig.savefig(f"{OUT}/graphique_1.png", dpi=300, bbox_inches="tight")
plt.show()

In [ ]:
fig, axes = plt.subplots(2, 3, figsize=(14, 8), constrained_layout=True)
cmap_ratio = plt.get_cmap("plasma_r").copy()
cmap_ratio.set_bad("#222222")

ratio_q = fv_q / np.where(it_q > 0, it_q, np.nan)
ratio_r = fv_r / np.where(it_r > 0, it_r, np.nan)

vmax_ratio_q = np.nanmax(ratio_q)
vmax_ratio_r = np.nanmax(ratio_r) if np.any(np.isfinite(ratio_r)) else 10

ims_rq, ims_rr = [], []
for ki, k in enumerate(kappa_values):
    ax = axes[0, ki]
    im = ax.pcolormesh(
        A_grid,
        B_grid,
        np.ma.masked_invalid(ratio_q[ki]),
        cmap=cmap_ratio,
        norm=LogNorm(vmin=1, vmax=vmax_ratio_q),
        shading="nearest",
    )
    ims_rq.append(im)
    ax.set_title(rf"Quadratique, $\kappa={k}$")
    ax.set_xlabel(r"$\alpha$")
    ax.set_ylabel(r"$\beta$")

for gi, g in enumerate(gamma_values):
    ax = axes[1, gi]
    if np.all(np.isnan(ratio_r[gi])):
        ax.set_facecolor("#222222")
        ax.set_title(rf"Rosenbrock, $\gamma={g}$ (échec)")
    else:
        im = ax.pcolormesh(
            A_grid,
            B_grid,
            np.ma.masked_invalid(ratio_r[gi]),
            cmap=cmap_ratio,
            norm=LogNorm(vmin=1, vmax=vmax_ratio_r),
            shading="nearest",
        )
        ims_rr.append(im)
        ax.set_title(rf"Rosenbrock, $\gamma={g}$")
    ax.set_xlabel(r"$\alpha$")
    ax.set_ylabel(r"$\beta$")

cb_rq = fig.colorbar(ims_rq[-1], ax=axes[0, :], shrink=0.85, pad=0.02)
cb_rq.set_label("Évaluations de $f$ par itération (log)")
if ims_rr:
    cb_rr = fig.colorbar(ims_rr[-1], ax=axes[1, :], shrink=0.85, pad=0.02)
    cb_rr.set_label("Évaluations de $f$ par itération (log)")

fig.suptitle(
    r"Surcoût induit par Armijo (appels à $f$ par itération)",
    fontsize=14,
    weight="bold",
)
fig.legend(
    handles=[
        plt.Line2D([0], [0], color="#222222", marker="s", linestyle="", markersize=8)
    ],
    labels=["Zones noires = non-convergence"],
    loc="upper right",
)
fig.savefig(f"{OUT}/graphique_2.png", dpi=300, bbox_inches="tight")
plt.show()

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(13, 6), constrained_layout=True)
markers = ["o", "s", "^"]

ax = axes[0]
for ki, k in enumerate(kappa_values):
    x = it_q[ki].flatten()
    y = fv_q[ki].flatten()
    c = B_grid.flatten()
    mask = ~np.isnan(x) & ~np.isnan(y)
    sc = ax.scatter(
        x[mask],
        y[mask],
        c=c[mask],
        cmap="coolwarm",
        marker=markers[ki],
        s=40,
        edgecolors="k",
        linewidth=0.5,
        alpha=0.8,
        label=rf"$\kappa={k}$",
    )
ax.set_xscale("log")
ax.set_yscale("log")
ax.set_xlabel("Nombre total d'itérations")
ax.set_ylabel(r"Nombre total d'évaluations de $f$")
ax.set_title("Compromis sur fonctions quadratiques")
ax.grid(True, which="both", alpha=0.3, linestyle="--")
ax.legend(loc="upper right")

ax = axes[1]
for gi, g in enumerate(gamma_values):
    x = it_r[gi].flatten()
    y = fv_r[gi].flatten()
    c = B_grid.flatten()
    mask = ~np.isnan(x) & ~np.isnan(y)
    if np.any(mask):
        sc = ax.scatter(
            x[mask],
            y[mask],
            c=c[mask],
            cmap="coolwarm",
            marker=markers[gi],
            s=40,
            edgecolors="k",
            linewidth=0.5,
            alpha=0.8,
            label=rf"$\gamma={g}$",
        )
ax.set_xscale("log")
ax.set_yscale("log")
ax.set_xlabel("Nombre total d'itérations")
ax.set_ylabel(r"Nombre total d'évaluations de $f$")
ax.set_title("Compromis sur la fonction de Rosenbrock")
ax.grid(True, which="both", alpha=0.3, linestyle="--")
if ax.get_legend_handles_labels()[0]:
    ax.legend(loc="upper right")

cbar = fig.colorbar(sc, ax=axes, shrink=0.8, pad=0.03)
cbar.set_label(r"Valeur de l'hyperparamètre $\beta$")
fig.suptitle(
    r"Itérations et évaluations de $f$", fontsize=15, weight="bold"
)
fig.savefig(f"{OUT}/graphique_3.png", dpi=300, bbox_inches="tight")
plt.show()

In [ ]:
TRAJ_PAIRS = [
    (0.04, 0.22, r"$\beta$ faible (coût minimal)"),
    (0.25, 0.50, r"Boyd ($\alpha{=}0.25,\ \beta{=}0.5$)"),
    (0.10, 0.70, r"$\beta$ modéré"),
    (0.10, 0.92, r"$\beta$ élevé"),
]

In [ ]:
def _run_quad_traj(ki, pi, si, alpha, beta, kappa, seed, dim, eps, max_iter):
    A, b = create_system(dim, cond=kappa, seed=seed)
    x_exact = np.linalg.solve(A, b)
    f = lambda x, A=A, b=b: (x.T @ A @ x) / 2 - b.T @ x
    gf = lambda x, A=A, b=b: grad_quadraticn(x, A, b)
    xk = np.zeros(dim)
    errors = [float(np.linalg.norm(xk - x_exact))]
    fk = f(xk)
    for _ in range(max_iter):
        gk = gf(xk)
        if np.linalg.norm(gk) < eps:
            break
        dk = -gk
        eta, fk, xk = armijo_fast(xk, fk, f, dk, float(np.dot(gk, dk)), alpha, beta)
        errors.append(float(np.linalg.norm(xk - x_exact)))
    return (ki, pi, si, errors)

In [ ]:
def _run_rosen_traj(gi, pi, si, alpha, beta, gamma, x0, eps, max_iter):
    f = lambda x, g=gamma: Rosenbrock(x, gamma=g)
    gf = lambda x, g=gamma: grad_rosenbrock(x, gamma=g)
    x_star = np.array([1.0, 1.0])
    xk = x0.copy()
    errors = [float(np.linalg.norm(xk - x_star))]
    fk = f(xk)
    for _ in range(max_iter):
        gk = gf(xk)
        if np.linalg.norm(gk) < eps:
            break
        dk = -gk
        eta, fk, xk = armijo_fast(xk, fk, f, dk, float(np.dot(gk, dk)), alpha, beta)
        errors.append(float(np.linalg.norm(xk - x_star)))
    return (gi, pi, si, errors)


In [ ]:
TRAJ_PATH = "../results_traj.npy"

if os.path.exists(TRAJ_PATH):
    traj_data = np.load(TRAJ_PATH)
    print(f"Chargé depuis le cache : {traj_data.shape}")
else:
    tasks_traj = [
        (ki, pi, si, alpha, beta, kappa, seeds[si], dim, eps, max_iter)
        for ki, kappa in enumerate(kappa_values)
        for pi, (alpha, beta, _) in enumerate(TRAJ_PAIRS)
        for si in range(len(seeds))
    ]
    raw_traj = list(
        tqdm(
            Parallel(n_jobs=-1, return_as="generator")(
                delayed(_run_quad_traj)(*t) for t in tasks_traj
            ),
            total=len(tasks_traj),
            desc="trajectories",
        )
    )
    max_len = max(len(r[3]) for r in raw_traj)
    traj_data = np.full(
        (len(kappa_values), len(TRAJ_PAIRS), len(seeds), max_len), np.nan
    )
    for ki, pi, si, errors in raw_traj:
        traj_data[ki, pi, si, : len(errors)] = errors
    np.save(TRAJ_PATH, traj_data)
    print(f"Sauvegardé : {traj_data.shape}")

In [ ]:
colors = ["#1f77b4", "#d62728", "#2ca02c", "#ff7f0e"]
linestyles = ["-", "--", "-.", ":"]

PAIR_SHORT = [r"$\beta$ faible", "Boyd", r"$\beta$ modéré", r"$\beta$ élevé"]

fig = plt.figure(figsize=(15, 6), constrained_layout=True)
gs = fig.add_gridspec(2, 3, height_ratios=[5, 1])

for ki, kappa in enumerate(kappa_values):
    ax = fig.add_subplot(gs[0, ki])
    ax_rug = fig.add_subplot(gs[1, ki], sharex=ax)

    panel_xmax = 0
    for pi, (alpha, beta, label) in enumerate(TRAJ_PAIRS):
        trajs = traj_data[ki, pi]
        lens = (~np.isnan(trajs)).sum(axis=1)
        max_common = int(np.percentile(lens, 50))
        panel_xmax = max(panel_xmax, max_common)

    for pi, (alpha, beta, label) in enumerate(TRAJ_PAIRS):
        trajs = traj_data[ki, pi]
        lens = (~np.isnan(trajs)).sum(axis=1)
        max_common = int(np.percentile(lens, 50))
        clipped = trajs[:, :max_common]
        with warnings.catch_warnings():
            warnings.simplefilter("ignore", RuntimeWarning)
            med = np.nanmedian(clipped, axis=0)
            q25 = np.nanpercentile(clipped, 25, axis=0)
            q75 = np.nanpercentile(clipped, 75, axis=0)
        iters = np.arange(max_common)
        mask = med > 0
        ax.semilogy(
            iters[mask],
            med[mask],
            color=colors[pi],
            ls=linestyles[pi],
            lw=1.8,
            label=label,
        )
        ax.fill_between(iters[mask], q25[mask], q75[mask], color=colors[pi], alpha=0.15)

    rho = (kappa - 1) / (kappa + 1)
    x0_norm = float(np.nanmedian(traj_data[ki, :, :, 0]))
    ref_iters = np.arange(panel_xmax)
    ax.semilogy(
        ref_iters,
        x0_norm * rho**ref_iters,
        color="black",
        ls=":",
        lw=1.2,
        alpha=0.6,
        label=rf"Théorique $\rho^k$",
    )

    ax.set_title(rf"$\kappa = {kappa}\ (\rho = {rho:.3f})$")
    if ki == 0:
        ax.set_ylabel(r"$\|x_k - x^\star\|_2$")
    ax.grid(True, which="major", alpha=0.3, ls="--")
    ax.tick_params(axis="x", labelbottom=False)
    ax.set_xlim(0, panel_xmax)
    if ki == 0:
        ax.legend(fontsize=8.5)

    for pi, (alpha, beta, _) in enumerate(TRAJ_PAIRS):
        trajs = traj_data[ki, pi]
        lens = (~np.isnan(trajs)).sum(axis=1) - 1
        in_view = (lens < max_iter) & (lens <= panel_xmax)
        out_view = ~in_view
        ax_rug.scatter(
            lens[in_view],
            np.full(in_view.sum(), pi),
            marker="|",
            s=80,
            color=colors[pi],
            alpha=0.8,
            linewidths=1.4,
        )
        if out_view.any():
            ax_rug.scatter(
                np.full(out_view.sum(), panel_xmax),
                np.full(out_view.sum(), pi),
                marker="x",
                s=22,
                color=colors[pi],
                alpha=0.7,
                linewidths=1.0,
                clip_on=False,
            )

    ax_rug.set_yticks(range(len(TRAJ_PAIRS)))
    if ki == 0:
        ax_rug.set_yticklabels(PAIR_SHORT, fontsize=8)
        ax_rug.set_ylabel("conv.", fontsize=9)
    else:
        ax_rug.set_yticklabels([])
    ax_rug.set_ylim(-0.5, len(TRAJ_PAIRS) - 0.5)
    ax_rug.invert_yaxis()
    ax_rug.set_xlabel("Itérations")
    ax_rug.set_xlim(0, panel_xmax)
    ax_rug.grid(False)
    for spine in ("top", "right"):
        ax_rug.spines[spine].set_visible(False)

fig.suptitle(
    "Convergence DG+Armijo sur fonction quadratique",
    fontsize=12,
    weight="bold",
)
fig.savefig(f"{OUT}/graphique_4.png", dpi=300, bbox_inches="tight")
plt.show()


In [ ]:
TRAJ_ROSEN_PATH = "../results_traj_rosen.npy"

rng_traj = np.random.default_rng(42)
x0_traj = rng_traj.uniform(x0_domain[0], x0_domain[1], size=(len(seeds), 2))

if os.path.exists(TRAJ_ROSEN_PATH):
    traj_rosen = np.load(TRAJ_ROSEN_PATH)
    print(f"Charge depuis le cache : {traj_rosen.shape}")
else:
    tasks_traj_r = [
        (gi, pi, si, alpha, beta, gamma, x0_traj[si], eps, max_iter)
        for gi, gamma in enumerate(gamma_values)
        for pi, (alpha, beta, _) in enumerate(TRAJ_PAIRS)
        for si in range(len(seeds))
    ]
    raw_traj_r = list(
        tqdm(
            Parallel(n_jobs=-1, return_as="generator")(
                delayed(_run_rosen_traj)(*t) for t in tasks_traj_r
            ),
            total=len(tasks_traj_r),
            desc="rosen trajectories",
        )
    )
    max_len_r = max(len(r[3]) for r in raw_traj_r)
    traj_rosen = np.full(
        (len(gamma_values), len(TRAJ_PAIRS), len(seeds), max_len_r), np.nan
    )
    for gi, pi, si, errors in raw_traj_r:
        traj_rosen[gi, pi, si, : len(errors)] = errors
    np.save(TRAJ_ROSEN_PATH, traj_rosen)
    print(f"Sauvegarde : {traj_rosen.shape}")


PAIR_SHORT = [r"$\beta$ faible", "Boyd", r"$\beta$ modéré", r"$\beta$ élevé"]

fig = plt.figure(figsize=(15, 6), constrained_layout=True)
gs = fig.add_gridspec(2, 3, height_ratios=[5, 1])

for gi, gamma in enumerate(gamma_values):
    ax = fig.add_subplot(gs[0, gi])
    ax_rug = fig.add_subplot(gs[1, gi], sharex=ax)

    for pi, (alpha, beta, label) in enumerate(TRAJ_PAIRS):
        trajs = traj_rosen[gi, pi]
        lens = (~np.isnan(trajs)).sum(axis=1)
        max_common = int(np.percentile(lens, 50))
        if max_common < 2:
            continue
        clipped = trajs[:, :max_common]
        with warnings.catch_warnings():
            warnings.simplefilter("ignore", RuntimeWarning)
            med = np.nanmedian(clipped, axis=0)
            q25 = np.nanpercentile(clipped, 25, axis=0)
            q75 = np.nanpercentile(clipped, 75, axis=0)
        iters = np.arange(max_common)
        mask = med > 0
        ax.semilogy(
            iters[mask],
            med[mask],
            color=colors[pi],
            ls=linestyles[pi],
            lw=1.8,
            label=label,
        )
        ax.fill_between(
            iters[mask], q25[mask], q75[mask], color=colors[pi], alpha=0.15
        )

    ax.set_title(rf"$\gamma = {gamma}$")
    if gi == 0:
        ax.set_ylabel(r"$\|x_k - x^\star\|_2$")
    ax.grid(True, which="major", alpha=0.3, ls="--")
    ax.tick_params(axis="x", labelbottom=False)
    if gi == 0:
        ax.legend(fontsize=8.5, loc="upper right")

    for pi, (alpha, beta, _) in enumerate(TRAJ_PAIRS):
        trajs = traj_rosen[gi, pi]
        lens = (~np.isnan(trajs)).sum(axis=1) - 1
        converged = lens < max_iter
        ax_rug.scatter(
            lens[converged],
            np.full(converged.sum(), pi),
            marker="|",
            s=80,
            color=colors[pi],
            alpha=0.8,
            linewidths=1.4,
        )
        if (~converged).any():
            ax_rug.scatter(
                lens[~converged],
                np.full((~converged).sum(), pi),
                marker="x",
                s=20,
                color=colors[pi],
                alpha=0.6,
                linewidths=1.0,
            )

    ax_rug.set_yticks(range(len(TRAJ_PAIRS)))
    if gi == 0:
        ax_rug.set_yticklabels(PAIR_SHORT, fontsize=8)
        ax_rug.set_ylabel("conv.", fontsize=9)
    else:
        ax_rug.set_yticklabels([])
    ax_rug.set_ylim(-0.5, len(TRAJ_PAIRS) - 0.5)
    ax_rug.invert_yaxis()
    ax_rug.set_xlabel("Itérations")
    ax_rug.grid(False)
    for spine in ("top", "right"):
        ax_rug.spines[spine].set_visible(False)

fig.suptitle(
    "Convergence DG+Armijo sur la fonction de Rosenbrock",
    fontsize=12,
    weight="bold",
)
fig.savefig(f"{OUT}/graphique_5.png", dpi=300, bbox_inches="tight")
plt.show()
